> !!! WORK IN PROGRESS !!!

# 0. Summary
- Download complete reference sites data (6424).
- Drop useless columns: 
    - dates: 
        - date_maj_station,
        - date_maj_ref_alti_station,
        - date_debut_ref_alti_station,
        - date_activation_ref_alti_station
    - "loi station" info:
        - type_contexte_loi_stat_station
        - type_loi_station
- Remove stations that are out-of-order on the configuration data time period. 
- Save to LOCAL_DATA_DIRECTORY as "hubeau_{HUBEAU_DATA_ENDPOINT}_1.csv". 

In [173]:
import os
import pandas
import datetime

# LOCAL COMPONENTS:
from config import LOCAL_DATA_DIRECTORY, HUBEAU_BASE_URL
from config import DATA_TIME_PERIOD, MAX_TRAINING_DATA_DATE, CONFIGURATION_DATA_DATE_FORMAT
from utils import request_json_all, save_dataframe_as_csv

# 1. Download data

In [174]:
HUBEAU_DATA_ENDPOINT = "referentiel/stations"

In [175]:
url = f"{HUBEAU_BASE_URL}/{HUBEAU_DATA_ENDPOINT}"
params = {"size": 200}  
results = request_json_all(url, params=params, timeout_in_seconds=60)

>>> page 001: 200 records / 6424 at rate 3.852866725487098 requests / second
>>> page 002: 200 records / 6424 at rate 1.9300175631598249 requests / second
>>> page 003: 200 records / 6424 at rate 2.313420693239645 requests / second
>>> page 004: 200 records / 6424 at rate 2.585245229899395 requests / second
>>> page 005: 200 records / 6424 at rate 2.7798286735991886 requests / second
>>> page 006: 200 records / 6424 at rate 2.955937320334435 requests / second
>>> page 007: 200 records / 6424 at rate 3.089183400846701 requests / second
>>> page 008: 200 records / 6424 at rate 1.6279940081680528 requests / second
>>> page 009: 200 records / 6424 at rate 1.5217812551043077 requests / second
>>> page 010: 200 records / 6424 at rate 1.618627947359629 requests / second
>>> page 011: 200 records / 6424 at rate 1.7185547829176278 requests / second
>>> page 012: 200 records / 6424 at rate 1.811496298962103 requests / second
>>> page 013: 200 records / 6424 at rate 1.8976240579026404 requests / 

In [176]:
print(type(results), results.keys())
assert(all(key in results.keys() for key in ("api_version", "count", "data")))
data = results["data"]
print(type(data), len(data))
assert(isinstance(data, list))
assert(len(data) > 0)
data_0 = data[0]
print(type(data_0), data_0.keys())

<class 'dict'> dict_keys(['api_version', 'count', 'data'])
<class 'list'> 6424
<class 'dict'> dict_keys(['code_site', 'libelle_site', 'code_station', 'libelle_station', 'type_station', 'coordonnee_x_station', 'coordonnee_y_station', 'code_projection', 'longitude_station', 'latitude_station', 'influence_locale_station', 'commentaire_station', 'altitude_ref_alti_station', 'code_systeme_alti_site', 'code_commune_station', 'libelle_commune', 'code_departement', 'code_region', 'libelle_region', 'code_cours_eau', 'libelle_cours_eau', 'uri_cours_eau', 'descriptif_station', 'date_maj_station', 'date_ouverture_station', 'date_fermeture_station', 'commentaire_influence_locale_station', 'code_regime_station', 'qualification_donnees_station', 'code_finalite_station', 'type_contexte_loi_stat_station', 'type_loi_station', 'code_sandre_reseau_station', 'date_debut_ref_alti_station', 'date_activation_ref_alti_station', 'date_maj_ref_alti_station', 'libelle_departement', 'en_service', 'geometry'])


In [177]:
df = pandas.DataFrame(data)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6424 entries, 0 to 6423
Data columns (total 39 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   code_site                             6424 non-null   str    
 1   libelle_site                          6424 non-null   str    
 2   code_station                          6424 non-null   str    
 3   libelle_station                       6424 non-null   str    
 4   type_station                          6424 non-null   str    
 5   coordonnee_x_station                  6424 non-null   float64
 6   coordonnee_y_station                  6424 non-null   float64
 7   code_projection                       6424 non-null   int64  
 8   longitude_station                     6424 non-null   float64
 9   latitude_station                      6424 non-null   float64
 10  influence_locale_station              5330 non-null   float64
 11  commentaire_station         

In [178]:
display(df.head())

,code_site,libelle_site,code_station,libelle_station,type_station,coordonnee_x_station,coordonnee_y_station,code_projection,longitude_station,latitude_station,...,code_finalite_station,type_contexte_loi_stat_station,type_loi_station,code_sandre_reseau_station,date_debut_ref_alti_station,date_activation_ref_alti_station,date_maj_ref_alti_station,libelle_departement,en_service,geometry
0,10110001,La Grande Rivière à Goyaves à Petit-Bourg [Bar...,1011000101,La Grande Rivière à Goyaves à Petit-Bourg [Bar...,STD,643352.000000,1.790354e+06,39,-61.658990,16.189402,...,1,NaN,NaN,"[BSH164, POH129, POH208, POH54]",2012-11-01T00:00:00Z,2017-07-11T00:00:00Z,2021-10-11T00:00:00Z,GUADELOUPE,True,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."
1,10110002,[Grande riviere a goyave] à Petit-Bourg [canal...,1011000201,[Grande riviere a goyave] à Petit-Bourg [canal...,STD,644100.000000,1.791070e+06,39,-61.651950,16.195829,...,0,NaN,NaN,None,NaN,NaN,NaN,GUADELOUPE,False,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."
2,10110003,[La Grande Rivière à Goyaves à Petit-Bourg-Bar...,1011000301,[La Grande Rivière à Goyaves à Petit-Bourg-Bar...,STD,643710.000000,1.790860e+06,39,-61.655610,16.193954,...,0,NaN,NaN,None,NaN,NaN,NaN,GUADELOUPE,False,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."
3,10110004,La Grande Rivière à Goyaves à Petit-Bourg [Ver...,1011000401,La Grande Rivière à Goyaves à Petit-Bourg [Ver...,STD,641900.000000,1.786900e+06,39,-61.672778,16.158271,...,0,NaN,NaN,None,NaN,NaN,NaN,GUADELOUPE,False,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."
4,10150001,La rivière Bras David à Petit-Bourg [Maison Fo...,1015000101,La rivière Bras David à Petit-Bourg [Maison Fo...,STD,639703.868935,1.788847e+06,39,-61.693200,16.175998,...,NaN,NaN,NaN,"[BSH164, POH208, POH129, POH54]",2001-07-01T00:00:00Z,2022-03-15T00:00:00Z,2022-04-25T00:00:00Z,GUADELOUPE,True,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."


# 2. Drop dates

In [179]:
# drop update dates
df.drop(columns=["date_maj_station"], inplace=True)
df.drop(columns=["date_maj_ref_alti_station"], inplace=True)    
df.drop(columns=["date_debut_ref_alti_station"], inplace=True) 
df.drop(columns=["date_activation_ref_alti_station"], inplace=True) 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6424 entries, 0 to 6423
Data columns (total 35 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   code_site                             6424 non-null   str    
 1   libelle_site                          6424 non-null   str    
 2   code_station                          6424 non-null   str    
 3   libelle_station                       6424 non-null   str    
 4   type_station                          6424 non-null   str    
 5   coordonnee_x_station                  6424 non-null   float64
 6   coordonnee_y_station                  6424 non-null   float64
 7   code_projection                       6424 non-null   int64  
 8   longitude_station                     6424 non-null   float64
 9   latitude_station                      6424 non-null   float64
 10  influence_locale_station              5330 non-null   float64
 11  commentaire_station         

In [180]:
display(df["type_station"].value_counts())

type_station
STD          6020
DEB           295
H              88
FICTIVE        12
LIMNIMERE       6
HC              2
LIMNI           1
Name: count, dtype: int64

# 3. Remove out-of-order stations 

In [181]:
display(df[["en_service", "date_ouverture_station", "date_fermeture_station"]])

,en_service,date_ouverture_station,date_fermeture_station
0,True,2012-11-11T00:00:00Z,NaN
1,False,1974-04-29T00:00:00Z,1975-12-03T00:00:00Z
2,False,1973-03-22T00:00:00Z,2005-10-31T00:00:00Z
3,False,1989-06-21T00:00:00Z,1992-01-01T00:00:00Z
4,True,2001-07-01T00:00:00Z,NaN
...,...,...,...
6419,False,1971-06-01T00:00:00Z,1989-04-01T00:00:00Z
6420,True,2020-06-17T00:00:00Z,NaN
6421,False,1969-01-01T00:00:00Z,1980-12-01T00:00:00Z
6422,False,1983-11-16T00:00:00Z,1988-05-01T00:00:00Z


In [182]:
df["date_ouverture_station"] = pandas.to_datetime(df["date_ouverture_station"])
df["date_fermeture_station"] = pandas.to_datetime(df["date_fermeture_station"])

In [183]:
min_configuration_date = pandas.Timestamp(DATA_TIME_PERIOD[0], tz="UTC")
max_configuration_date = pandas.Timestamp(MAX_TRAINING_DATA_DATE, tz="UTC")
print("min configuration date:", min_configuration_date, "/", "max configuration date:", max_configuration_date)
max_closure_date = df[df["en_service"] == False]["date_fermeture_station"].max()
print("max closure date:", max_closure_date)

min configuration date: 2007-01-01 00:00:00+00:00 / max configuration date: 2025-12-31 00:00:00+00:00
max closure date: 2026-05-11 00:00:00+00:00


In [184]:
assert(len(df[(df["en_service"] == True) & (~df["date_fermeture_station"].isna())]) == 0)
assert(len(df[df["date_fermeture_station"] < df["date_ouverture_station"]]) == 0)

In [185]:
df = df[(df["date_ouverture_station"] < min_configuration_date) & ((df["date_fermeture_station"].isna()) | (df["date_fermeture_station"] > max_configuration_date))]
df.drop(columns=["en_service", "date_ouverture_station"])
df.info()

<class 'pandas.DataFrame'>
Index: 2534 entries, 4 to 6408
Data columns (total 35 columns):
 #   Column                                Non-Null Count  Dtype              
---  ------                                --------------  -----              
 0   code_site                             2534 non-null   str                
 1   libelle_site                          2534 non-null   str                
 2   code_station                          2534 non-null   str                
 3   libelle_station                       2534 non-null   str                
 4   type_station                          2534 non-null   str                
 5   coordonnee_x_station                  2534 non-null   float64            
 6   coordonnee_y_station                  2534 non-null   float64            
 7   code_projection                       2534 non-null   int64              
 8   longitude_station                     2534 non-null   float64            
 9   latitude_station                   

# 4. Explore and prune other columns

In [186]:
df.drop(columns=["type_contexte_loi_stat_station"], inplace=True)
df.drop(columns=["type_loi_station"], inplace=True)

In [187]:
display(df["commentaire_influence_locale_station"].value_counts())
display(df["influence_locale_station"].value_counts())

commentaire_influence_locale_station
Attention: il s'agit d'un débit naturel reconstitué (QNR)                                    21
Ouvrage de prise                                                                              9
Marée.                                                                                        7
marée                                                                                         4
Débits désinfluencés.                                                                         3
                                                                                             ..
Par pompages en amont                                                                         1
Influence forte des barrages de baigneurs en été, corrigée lors du traitement des données     1
Voir site                                                                                     1
Pompages amont                                                                                1
Pom

influence_locale_station
1.0    1383
2.0     407
3.0     336
0.0     194
4.0       9
Name: count, dtype: int64

In [188]:
display(df["descriptif_station"].value_counts())
display(df["type_station"].value_counts())
display(df["qualification_donnees_station"].value_counts())

descriptif_station
EDF                      16
HE                       14
Barrage (aval) [VNF]     11
Barrage (amont) [VNF]    10
CNR                      10
                         ..
Pont du Faïo              1
site barrage              1
Pont d'Acitaja            1
Sampolo                   1
Canniciu                  1
Name: count, Length: 500, dtype: int64

type_station
STD          2452
DEB            65
H              10
FICTIVE         4
LIMNIMERE       3
Name: count, dtype: int64

qualification_donnees_station
20    1651
16     641
12     242
Name: count, dtype: int64

In [189]:
display(df["libelle_site"].value_counts())
display(df["libelle_commune"].value_counts())
display(df["libelle_departement"].value_counts())
display(df["libelle_region"].value_counts())
display(df["libelle_cours_eau"].value_counts())

libelle_site
La Loire à Chalonnes-sur-Loire et à Ingrandes et à Montjean-sur-Loire et à Saint-Florent-le-Vieil    4
L'Yerres à Boussy-Saint-Antoine                                                                      3
La Seine à Alfortville et à Villeneuve-Saint-Georges et à Vitry-sur-Seine                            3
La Marne à Meaux                                                                                     3
La Marne à Saint-Maurice                                                                             3
                                                                                                    ..
L'Alesani à Pietra-di-Verde                                                                          1
La Bravona à Tallone [Site barrage]                                                                  1
Le Fium Alto à Taglio-Isolaccio                                                                      1
L'U Fium'Orbu à Ghisoni [Sampolo]                           

libelle_commune
SAINT-JOSEPH        5
SAINT-PAUL          5
BANDRABOUA          4
SAINT-DENIS         4
NOGENT-SUR-SEINE    4
                   ..
PIETRA-DI-VERDE     1
TALLONE             1
TAGLIO-ISOLACCIO    1
GHISONI             1
SARI-SOLENZARA      1
Name: count, Length: 2194, dtype: int64

libelle_departement
AVEYRON                  70
LOZERE                   54
SEINE-ET-MARNE           53
FINISTERE                47
PYRENEES-ORIENTALES      47
                         ..
TERRITOIRE DE BELFORT     6
CORSE-DU-SUD              6
PARIS                     2
HAUTS-DE-SEINE            2
SEINE-SAINT-DENIS         1
Name: count, Length: 101, dtype: int64

libelle_region
OCCITANIE                     504
AUVERGNE-RHONE-ALPES          332
NOUVELLE-AQUITAINE            283
GRAND EST                     264
BOURGOGNE-FRANCHE-COMTE       208
PAYS DE LA LOIRE              158
BRETAGNE                      135
PROVENCE-ALPES-COTE D'AZUR    119
ILE-DE-FRANCE                 111
HAUTS-DE-FRANCE               109
NORMANDIE                     106
CENTRE-VAL DE LOIRE            94
LA REUNION                     29
CORSE                          19
GUYANE                         12
MARTINIQUE                     11
MAYOTTE                        10
GUADELOUPE                      9
Name: count, dtype: int64

libelle_cours_eau
La Seine              54
la Loire              51
Le Rhône              31
La Marne              28
La Garonne            22
                      ..
Rivière d'Alesani      1
Rivière de Bravona     1
Le Fium Alto           1
u Fium'Orbu            1
La Solenzara           1
Name: count, Length: 1151, dtype: int64

In [190]:
display(df[df["libelle_site"] == "L'Yerres à Yerres"])

,code_site,libelle_site,code_station,libelle_station,type_station,coordonnee_x_station,coordonnee_y_station,code_projection,longitude_station,latitude_station,...,date_ouverture_station,date_fermeture_station,commentaire_influence_locale_station,code_regime_station,qualification_donnees_station,code_finalite_station,code_sandre_reseau_station,libelle_departement,en_service,geometry
1113,F4870008,L'Yerres à Yerres,F487000801,L'Yerres à Yerres - SR4 - Céravennes - Barrage...,STD,662829.9,6845846.6,26,2.494816,48.711687,...,2000-05-09 00:00:00+00:00,NaT,NaN,1,16,2,None,ESSONNE,True,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."
1114,F4870008,L'Yerres à Yerres,F487000802,L'Yerres à Yerres - SR4 - Céravennes - Barrage...,STD,662816.7,6845885.2,26,2.494633,48.712033,...,2000-05-09 00:00:00+00:00,NaT,NaN,1,16,2,None,ESSONNE,True,"{'type': 'Point', 'crs': {'type': 'name', 'pro..."


# 10. Save the dataframe

In [191]:
df.info()

<class 'pandas.DataFrame'>
Index: 2534 entries, 4 to 6408
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype              
---  ------                                --------------  -----              
 0   code_site                             2534 non-null   str                
 1   libelle_site                          2534 non-null   str                
 2   code_station                          2534 non-null   str                
 3   libelle_station                       2534 non-null   str                
 4   type_station                          2534 non-null   str                
 5   coordonnee_x_station                  2534 non-null   float64            
 6   coordonnee_y_station                  2534 non-null   float64            
 7   code_projection                       2534 non-null   int64              
 8   longitude_station                     2534 non-null   float64            
 9   latitude_station                   

In [192]:
display(df.describe())

,coordonnee_x_station,coordonnee_y_station,code_projection,longitude_station,latitude_station,influence_locale_station,altitude_ref_alti_station,code_systeme_alti_site,code_regime_station,qualification_donnees_station
count,2.534000e+03,2.534000e+03,2534.000000,2534.000000,2534.000000,2329.000000,2.038000e+03,2038.000000,2534.000000,2534.000000
mean,6.679502e+05,6.460574e+06,26.013023,2.713060,44.877326,1.391584,1.026901e+03,2.555447,1.015785,18.224152
std,2.140740e+05,9.064566e+05,3.507961,9.743275,9.256383,0.846682,3.552926e+04,1.471582,0.158168,2.647368
min,2.479662e+00,4.283327e+01,5.000000,-61.782849,-21.381010,0.000000,-3.569000e+00,0.000000,1.000000,12.000000
25%,5.022200e+05,6.352818e+06,26.000000,0.488183,44.240463,1.000000,3.571500e+01,2.000000,1.000000,16.000000
50%,6.774865e+05,6.592028e+06,26.000000,2.829894,46.290379,1.000000,1.270000e+02,3.000000,1.000000,20.000000
75%,8.363205e+05,6.807336e+06,26.000000,4.928832,48.210365,2.000000,2.479175e+02,3.000000,1.000000,20.000000
max,1.233024e+06,8.594348e+06,41.000000,55.721977,50.920179,4.000000,1.603520e+06,31.000000,3.000000,20.000000


In [193]:
endpoint = HUBEAU_DATA_ENDPOINT.replace("/","_")
base_name = f"hubeau_{endpoint}_1"
filepath = save_dataframe_as_csv(df, base_name=base_name, output_dirpath=LOCAL_DATA_DIRECTORY)

In [194]:
filesize_in_kb = os.path.getsize(filepath) / 1024
print(f">>> saved : \"{filepath}\" ({len(df)} records, {filesize_in_kb:.0f} Kb)")

>>> saved : "c:\Users\nicolas\Repositories\Jedha\Shares\M11\ml-flood-forecasting\eda\data\hubeau\hubeau_referentiel_stations_1.csv" (2534 records, 1646 Kb)
